Mini projet QPE

In [ ]:
!pip install mpqp[all]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of amazon-braket-schemas to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of awscli to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of awscli to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://

In [ ]:
from mpqp import QCircuit, H, P, X, Ry, Rz, CZ, CNOT, CRk, CustomGate, Barrier, Gate, SWAP, IBMDevice, run, BasisMeasure, Z
from math import floor
import numpy as np

QFT

In [ ]:
class QFT(QCircuit):

  def __init__(self, n_qubits: int):

     super().__init__(n_qubits, nb_cbits=n_qubits)
     self._build()

  def _build(self):
      for j in range(self.nb_qubits):
          self.add(H(j))
          self.add([CRk(i-j+1, i, j) for i in range(j+1, self.nb_qubits)])
          self.add(Barrier())
      self.add([SWAP(i, self.nb_qubits-1-i) for i in range(int(floor(self.nb_qubits / 2)))])

#Pour appeller la QFT inverse, juste créer une QFT et appeler la méthode .inverse() pour inverser le circuit



Prepare circuit

In [ ]:
def prepare_circuit(precision: int, eigenstate_circuit: QCircuit) -> QCircuit:
  circuit = QCircuit(precision + eigenstate_circuit.size()[0])
  circuit.append(eigenstate_circuit, qubits_offset=precision)
  circuit.add([H(i) for i in range(0, precision)])
  return circuit

def prepare_circuit_test():
  eigenstate_circuit = QCircuit(2)
  eigenstate_circuit.add([X(0), X(1)])
  prepare_circuit(5, eigenstate_circuit).pretty_print()


prepare_circuit_test()

QCircuit : Size (Qubits, Cbits) = (7, 0), Nb instructions = 7
     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     ├───┤
q_2: ┤ H ├
     ├───┤
q_3: ┤ H ├
     ├───┤
q_4: ┤ H ├
     ├───┤
q_5: ┤ X ├
     ├───┤
q_6: ┤ X ├
     └───┘


Decompose

In [ ]:
def decompose_CU(theta: float, phi: float, gamma: float, control: int, target: int) -> list[Gate]:
  """
  From https://arxiv.org/pdf/quant-ph/9503016
  CU = A CX B CX C P
  """
  return [
      P((phi + gamma) / 2, control),
      Rz((gamma - phi) / 2, target),
      CNOT(control, target),
      Rz((phi + gamma) / -2, target), Ry(-theta / 2, target),
      CNOT(control, target),
      Ry(theta / 2, target), Rz(phi, target),
      ]

def test_decomposition(theta=np.pi/3, phi=np.pi/4, gamma=np.pi/5) -> None:
    gates = decompose_CU(theta, phi, gamma, 0, 1)
    CU_circuit = QCircuit(2)
    CU_circuit.add(gates)
    CU_decomposed = CU_circuit.to_gate().to_matrix()

    I = np.eye(2)
    U = np.array([
        [np.cos(theta/2), -np.exp(1j*gamma) * np.sin(theta/2)],
        [np.exp(1j*phi) * np.sin(theta/2), np.exp(1j*(phi+gamma)) * np.cos(theta/2)]
    ])
    CU_target = np.block([[I, np.zeros((2,2))],
                          [np.zeros((2,2)), U]])

    is_close = np.allclose(CU_target, CU_decomposed)
    assert is_close, "The matrices do not match."

test_decomposition()

In [ ]:
def controlled_u(circuit: QCircuit, unitary: Gate, precision: int) -> QCircuit:
    for j in range(precision):
        # carefull handling of bit weight ordering
        power = 2**(precision - 1 - j)

        # quick exponentiation
        U_mat = np.linalg.matrix_power(unitary.to_matrix(), power)

        theta, phi, gamma = extract_params(U_mat)
        circuit.add(decompose_CU(theta, phi, gamma, control=j, target=precision))

    return circuit

def extract_params(U_mat: np.ndarray) -> tuple[float, float, float]:
    a = U_mat[0, 0] # cos(theta/2)
    b = U_mat[0, 1] # -e^(i gamma) sin(theta/2)
    c = U_mat[1, 0] # e^(i phi) sin(theta/2)

    theta = 2 * np.arccos(np.clip(np.abs(a), 0, 1)) # np.clip agains floating point inaccuracies

    if np.isclose(np.sin(theta / 2), 0):
      # special handling because it may cause noisy phi and gamma phases
        phi = 0.0
        gamma = np.angle(U_mat[1, 1] / (U_mat[0, 0] + 1e-12))
    else:
        phi = np.angle(c) - np.angle(a)
        gamma = np.angle(-b) - np.angle(a)

    return theta, phi, gamma

# Test
eigenstate_circuit = QCircuit(2)
eigenstate_circuit.add([X(0), X(1)])
a = prepare_circuit(2, eigenstate_circuit)

U = Z(0)

result = controlled_u(a, U, precision=2)
result.pretty_print()


QCircuit : Size (Qubits, Cbits) = (4, 0), Nb instructions = 20
     ┌───┐ ┌──────┐                                                          »
q_0: ┤ H ├─┤ P(0) ├───■──────────────────────■───────────────────────────────»
     ├───┤┌┴──────┴┐  │                      │                               »
q_1: ┤ H ├┤ P(π/2) ├──┼──────────────────────┼───────────────────────────────»
     ├───┤├───────┬┘┌─┴─┐┌───────┐┌───────┐┌─┴─┐┌───────┐┌───────┐┌─────────┐»
q_2: ┤ X ├┤ Rz(0) ├─┤ X ├┤ Rz(0) ├┤ Ry(0) ├┤ X ├┤ Ry(0) ├┤ Rz(0) ├┤ Rz(π/2) ├»
     ├───┤└───────┘ └───┘└───────┘└───────┘└───┘└───────┘└───────┘└─────────┘»
q_3: ┤ X ├───────────────────────────────────────────────────────────────────»
     └───┘                                                                   »
«                                                      
«q_0: ─────────────────────────────────────────────────
«                                                      
«q_1: ──■─────────────────────────■────────────────────
« 

mesure

In [ ]:
def measure_circuit(circuit: QCircuit, precision: int):
    circuit.add([(BasisMeasure(
        list(range(precision)),
        list(range(precision))
    ))])
    return run(circuit, IBMDevice.AER_SIMULATOR)

def get_phase(result, precision: int) -> float:
    most_likely = max(result.samples, key=lambda s: s.count)

    k = int(most_likely.bin_str, 2)

    phase = 2 * np.pi * k / (2 ** precision)
    return phase


Algorithm for QPE

In [ ]:
def qpe(unitary: Gate, eigenstate_circuit: QCircuit, precision: int) -> float:
  circuit = prepare_circuit(precision, eigenstate_circuit)
  circuit = controlled_u(circuit, unitary, precision)
  circuit.append(QFT(precision).inverse())
  result = measure_circuit(circuit, precision)
  # print(circuit)
  return get_phase(result, precision)

In [ ]:
print("--- Test 1: Z-Gate (Phase pi) ---")
PRECISION_0 = 3

eigenstate_circuit_0 = QCircuit(1)
eigenstate_circuit_0.add(X(0))

U_0 = Z(0)
phase_0 = qpe(U_0, eigenstate_circuit_0, PRECISION_0)
print(f"Target: 1.0000 pi | Estimated: {phase_0/np.pi:.4f} pi")
print()



print("--- Test 2: T-Gate (Phase pi/4) ---")
PRECISION_1 = 3
eigenstate_circuit_1 = QCircuit(1)
eigenstate_circuit_1.add(X(0))

U_1 = P(np.pi/4, 0)
phase_1 = qpe(U_1, eigenstate_circuit_1, PRECISION_1)
print(f"Target: 0.2500 pi | Estimated: {phase_1/np.pi:.4f} pi")
print()



print("--- Test 3: S-Gate (Phase pi/2) ---")
PRECISION_2 = 2
eigenstate_circuit_2 = QCircuit(1)
eigenstate_circuit_2.add(X(0))

U_2 = P(np.pi/2, 0)
phase_2 = qpe(U_2, eigenstate_circuit_2, PRECISION_2)
print(f"Target: 0.5000 pi | Estimated: {phase_2/np.pi:.4f} pi")
print()


print("With 3 qubits, the closest fraction to 2/3 is 5/8 (0.625) or 6/8 (0.75).")
print()

print("--- Test 4: Imperfect Precision (Phase 2pi/3) ---")
PRECISION_3 = 3
eigenstate_circuit_3 = QCircuit(1)
eigenstate_circuit_3.add(X(0))

target_phase = 2 * np.pi / 3
U_3 = P(target_phase, 0)

phase_3 = qpe(U_3, eigenstate_circuit_3, PRECISION_3)
print(f"Target: {target_phase/np.pi:.4f} pi | Estimated: {phase_3/np.pi:.4f} pi")
print()

print("--- Test 5: Higher Precision (Phase 2pi/3) ---")
PRECISION_4 = 6
eigenstate_circuit_4 = QCircuit(1)
eigenstate_circuit_4.add(X(0))

U_4 = P(target_phase, 0)

phase_4 = qpe(U_4, eigenstate_circuit_4, PRECISION_4)
print(f"Target: {target_phase/np.pi:.4f} pi | Estimated: {phase_4/np.pi:.4f} pi")

--- Test 1: Z-Gate (Phase pi) ---
Target: 1.0000 pi | Estimated: 1.0000 pi

--- Test 2: T-Gate (Phase pi/4) ---
Target: 0.2500 pi | Estimated: 0.2500 pi

--- Test 3: S-Gate (Phase pi/2) ---
Target: 0.5000 pi | Estimated: 0.5000 pi

With 3 qubits, the closest fraction to 2/3 is 5/8 (0.625) or 6/8 (0.75).

--- Test 4: Imperfect Precision (Phase 2pi/3) ---
Target: 0.6667 pi | Estimated: 0.7500 pi

--- Test 5: Higher Precision (Phase 2pi/3) ---
Target: 0.6667 pi | Estimated: 0.6562 pi
